## Preparation

In [1]:
from stark_qa import load_skb

OUTPUT_DIR = "../graphs/stark-amazon"

dataset_name = "amazon"
skb = load_skb(dataset_name, download_processed=True)

Loading from /home/wagnerr/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/amazon/processed!
Loading cached graph with meta link types ['brand', 'category', 'color']


## Save relational information (triples)

In [2]:
# edge_index[0][x] returns ID of node 1, edge_index[1][x] returns ID of node 2 of edge x
src_ids, dst_ids = skb.edge_index.tolist()

# edge_types[x] returns type ID of edge x
edge_types = skb.edge_types.tolist()

# Translating edge type IDs to corresponding strings like 1 --> "also_view"
edge_names = [skb.edge_type_dict[e] for e in edge_types]

# Create tuples as (node 1 ID, edge name, node 2 ID)
all_tuples = list(zip(src_ids, edge_names, dst_ids))

# Save triples to .txt file
with open(f"{OUTPUT_DIR}/triples.txt", "w") as f:
    for t in all_tuples:
        f.write(f"{t[0]},{t[1]},{t[2]}\n")

## Save textual information (node properties)

In [3]:
import json

EXCLUDE_KEYS = ["brand", "category", "color"]

# Nodes as a list of node dictionaries
nodes_json = []
for n_id in range(skb.num_nodes()):
    node_dict = {
        "id": n_id,
        "type": skb.get_node_type_by_id(n_id),
    }

    # Add all node properties, excluding properties that are actually relations
    for key, value in skb.node_info[n_id].items():
        if key not in EXCLUDE_KEYS:
            node_dict[key] = str(value)

    nodes_json.append(node_dict)

# Save nodes to .json file
with open(f"{OUTPUT_DIR}/nodes.json", "w") as f:
    json.dump(nodes_json, f, indent=4)